# Day 13 - ResNet - 残差连接：为什么152层不会退化

> 目标：理解残差连接（Residual Connection / Skip Connection）如何解决深层网络的退化问题
>
> LeNet(7层) → AlexNet(8层) → VGG(16层) → **ResNet(152层, 参数更少!)**

---

## 回顾 Day12 的问题

```
VGG16: 16层, 138M参数
实验发现: 56层 比 20层 训练误差更高 -> 退化问题 (Degradation)
          不是过拟合, 是优化困难!
```

**ResNet = Residual Network** — 2015 ImageNet 冠军，何恺明团队

| 对比 | VGG16 | ResNet152 |
|------|-------|-----------|
| 层数 | 16 | **152** |
| 参数 | 138M | **25M** (少 5.5 倍!) |
| Top-5错误 | ~7.4% | **~3.6%** |

**关键问题:** 为什么 152 层比 16 层参数更少、效果更好？

> 答案: 去除巨大的全连接层 + 全局平均池化 (GAP) + 残差连接让梯度畅通

In [4]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import time

print('=' * 50)
print('Day 13 - ResNet: Residual Learning')
print('=' * 50)

Day 13 - ResNet: Residual Learning


---
## 1. 核心思想：残差学习 (Residual Learning)

### 问题

假设我们想让网络学习某个映射 $H(x)$。

如果堆叠了更多层，期望至少不差于浅层网络（多出的层学恒等映射 $H(x)=x$ 就行）。

**但实际**：深层网络很难拟合恒等映射，导致退化。

### 残差连接的直觉

```
传统: 层直接学 H(x)                        残差: 层学残差 F(x) = H(x) - x
  x -> [Conv] -> [ReLU] -> [Conv] -> H(x)     x -> [Conv] -> [ReLU] -> [Conv] -> F(x)
                                               |                                      |
                                               +----------> 加起来 (F(x) + x) <--------+
```

**为什么有效？**

- 如果 $H(x)=x$ 是最优解（恒等映射），残差 $F(x)=0$ 比直接学 $H(x)=x$ 容易得多
- 因为 $F(x)=0$ 只需把权重推向 0（有初始化保证），而 $H(x)=x$ 需要权重精确配平
- 梯度直接通过捷径 (shortcut) 反向传播到前层，**梯度消失问题大幅缓解**

### 数学形式

$$
y = F(x, \{W_i\}) + x
$$

其中 $F$ 是残差映射 (通常 2-3 层卷积)，$x$ 是恒等映射 (identity shortcut)

### BasicBlock - 基本残差块 (ResNet18/34)

```
结构:
    x -> [Conv 3x3] -> [BN] -> [ReLU] -> [Conv 3x3] -> [BN] -> + -> [ReLU]
      |                                                           |
      +---------------- 恒等连接 (identity shortcut) -------------+
```

当维度不匹配时（stride=2 或通道数变化），用 1x1 卷积调整 shortcut

In [5]:
class BasicBlock(nn.Module):
    """基本残差块: 两个 3x3 卷积, 用于 ResNet18/34"""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 维度不匹配时, 用 1x1 卷积调整
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)  # <--- 残差连接!
        out = F.relu(out)
        return out

# 验证
block = BasicBlock(64, 64)
x = torch.randn(1, 64, 32, 32)
out = block(x)
print(f'BasicBlock: {list(x.shape)} -> {list(out.shape)}')
print(f'  参数: {sum(p.numel() for p in block.parameters()):,}')

BasicBlock: [1, 64, 32, 32] -> [1, 64, 32, 32]
  参数: 73,984


### Bottleneck - 瓶颈块 (ResNet50/101/152)

```
结构: 1x1 -> 3x3 -> 1x1
    x -> [1x1, 压缩1/4] -> [3x3] -> [1x1, 扩展x4] -> + -> [ReLU]
      |                                                    |
      +-------------------- 恒等连接 ----------------------+

为什么叫 Bottleneck?
    输入256 -> 1x1 -> 64(瓶颈) -> 3x3 -> 64 -> 1x1 -> 256(恢复)
    中间 64 是瓶颈, 计算量只有直接两个3x3的 1/4!
```

In [6]:
class Bottleneck(nn.Module):
    """瓶颈残差块: 1x1 -> 3x3 -> 1x1, 用于 ResNet50+"""
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        bottleneck_channels = out_channels

        self.conv1 = nn.Conv2d(in_channels, bottleneck_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)
        self.conv2 = nn.Conv2d(bottleneck_channels, bottleneck_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)
        self.conv3 = nn.Conv2d(bottleneck_channels, out_channels * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, 1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)  # <--- 残差连接!
        out = F.relu(out)
        return out

# 验证
bottleneck = Bottleneck(256, 64)
x = torch.randn(1, 256, 16, 16)
out = bottleneck(x)
print(f'Bottleneck: {list(x.shape)} -> {list(out.shape)}')
print(f'  参数: {sum(p.numel() for p in bottleneck.parameters()):,}')

Bottleneck: [1, 256, 16, 16] -> [1, 256, 16, 16]
  参数: 70,400


---
## 2. ResNet 整体架构

```
ResNet 家族 (输入 224x224 图像)

         [7x7 Conv, 64, stride 2]          <- 开头一个大卷积
         [3x3 MaxPool, stride 2]           <- 快速降采样
    |--------------------------------|
    | Layer1: 56x56 |  x N         |  <- 通道: 64
    | Layer2: 28x28 |  x N         |  <- 通道: 128 (stride=2)
    | Layer3: 14x14 |  x N         |  <- 通道: 256 (stride=2)
    | Layer4:  7x7  |  x N         |  <- 通道: 512 (stride=2)
    |--------------------------------|
         [Global Avg Pooling (GAP)]        <- 代替 Flatten + FC!
         [FC 1000]                         <- 分类

网络     层数     block类型    block数 (L1-L4)   参数
ResNet18  18     BasicBlock   [2,2,2,2]       11M
ResNet34  34     BasicBlock   [3,4,6,3]       22M
ResNet50  50     Bottleneck   [3,4,6,3]       25M    <- 经典
ResNet101 101    Bottleneck   [3,4,23,3]      45M
ResNet152 152    Bottleneck   [3,8,36,3]      60M
```

**关键设计:**
- 没有 Dropout! (BN 已经够正则化)
- 没有巨大的全连接层! (GAP 代替)
- 每层通道数固定: 64->128->256->512

In [7]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 64

        # CIFAR-10 (32x32) 版本: 用 3x3 开头, 无 MaxPool
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        """创建一层残差块, 第一个 block 可能降采样"""
        layers = []
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


def ResNet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

def ResNet34(num_classes=10):
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes)

def ResNet50(num_classes=10):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)

In [8]:
# === 参数量对比: ResNet vs VGG ===

r18 = ResNet18()
r34 = ResNet34()
r50 = ResNet50()

print('=' * 40)
print('ResNet 参数量对比')
print('=' * 40)
print(f'ResNet18:  {sum(p.numel() for p in r18.parameters()):>8,} 参数 ({sum(p.numel() for p in r18.parameters())/1e6:.1f}M)')
print(f'ResNet34:  {sum(p.numel() for p in r34.parameters()):>8,} 参数 ({sum(p.numel() for p in r34.parameters())/1e6:.1f}M)')
print(f'ResNet50:  {sum(p.numel() for p in r50.parameters()):>8,} 参数 ({sum(p.numel() for p in r50.parameters())/1e6:.1f}M)')
print()
print(f'VGG13(轻量): 9.6M   (Day12 参考)')
print(f'ResNet18:     11.0M  (更深但参数差不多)')
print(f'-> 关键: GAP 代替了巨大全连接层!')

# 验证输入输出
x = torch.randn(4, 3, 32, 32)
out = r18(x)
print(f'\n输入: {list(x.shape)} -> 输出: {list(out.shape)}')

ResNet 参数量对比
ResNet18:  11,173,962 参数 (11.2M)
ResNet34:  21,282,122 参数 (21.3M)
ResNet50:  23,520,842 参数 (23.5M)

VGG13(轻量): 9.6M   (Day12 参考)
ResNet18:     11.0M  (更深但参数差不多)
-> 关键: GAP 代替了巨大全连接层!

输入: [4, 3, 32, 32] -> 输出: [4, 10]


---
## 3. 对照实验: Plain vs ResNet

复现经典实验, 验证退化:

| 网络 | 层数 | 有无残差连接 | 预期 |
|------|------|------------|------|
| PlainNet-20 | ~20层 | ❌ 无 | 基线 |
| PlainNet-32 | ~32层 | ❌ 无 | **退化!** 比20层差 |
| ResNet-18 | ~18层 | ✅ 有 | 基线 |
| ResNet-34 | ~34层 | ✅ 有 | **更好!** 比18层好 |

> **退化现象**: 32层 PlainNet 比 20层 PlainNet 更差 (优化困难)
> **残差解决**: 34层 ResNet 比 18层 ResNet 更好 (梯度畅通)

In [9]:
# PlainBlock: 和 BasicBlock 结构一样, 但没有残差连接!

class PlainBlock(nn.Module):
    """普通卷积块 (无残差连接) - 对照实验"""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        residual = self.downsample(x)
        out = out + residual  # 仅维度匹配, 无残差学习
        out = F.relu(out)
        return out

def PlainNet20(num_classes=10):
    return ResNet(PlainBlock, [2, 2, 2, 2], num_classes)

def PlainNet32(num_classes=10):
    return ResNet(PlainBlock, [3, 4, 6, 3], num_classes)

In [10]:
# 参数量完全一样, 唯一区别是有无残差连接

p20 = PlainNet20()
p32 = PlainNet32()

print(f'PlainNet20: {sum(p.numel() for p in p20.parameters()):>8,} 参数')
print(f'PlainNet32: {sum(p.numel() for p in p32.parameters()):>8,} 参数')
print(f'ResNet18:   {sum(p.numel() for p in r18.parameters()):>8,} 参数  (和 Plain20 一样)')
print(f'ResNet34:   {sum(p.numel() for p in r34.parameters()):>8,} 参数  (和 Plain32 一样)')
print()
print('结论: Plain vs ResNet 参数量一致, 唯一变量是残差连接')

PlainNet20: 11,173,962 参数
PlainNet32: 21,282,122 参数
ResNet18:   11,173,962 参数  (和 Plain20 一样)
ResNet34:   21,282,122 参数  (和 Plain32 一样)

结论: Plain vs ResNet 参数量一致, 唯一变量是残差连接


---
## 4. 准备数据

> 优先加载 CIFAR-10, 失败则用随机数据 (确保 notebook 可运行)

In [11]:
import torch
from torch.utils.data import DataLoader, TensorDataset


def create_random_data(num_train=2000, num_test=500):
    """
    生成随机的训练和测试数据
    模拟 CIFAR-10 的数据格式: (N, 3, 32, 32) 的图像 + 0~9 的标签
    """
    # 生成随机训练数据: 2000张 3通道 32x32 的随机图像
    X_train = torch.randn(num_train, 3, 32, 32)
    # 生成随机训练标签: 2000个 0~9 之间的随机整数
    y_train = torch.randint(0, 10, (num_train,))

    # 生成随机测试数据: 500张 3通道 32x32 的随机图像
    X_test = torch.randn(num_test, 3, 32, 32)
    # 生成随机测试标签: 500个 0~9 之间的随机整数
    y_test = torch.randint(0, 10, (num_test,))

    # 将张量打包为 Dataset
    train_data = TensorDataset(X_train, y_train)
    test_data = TensorDataset(X_test, y_test)

    # 创建 DataLoader，训练集打乱，测试集不打乱
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

    return train_loader, test_loader


# 直接生成随机数据，不再尝试加载 CIFAR-10
train_loader, test_loader = create_random_data()
print(f'训练集: {len(train_loader.dataset)} 张 | 测试集: {len(test_loader.dataset)} 张')

训练集: 2000 张 | 测试集: 500 张


---
## 5. 训练函数

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
from torch.utils.data import DataLoader, TensorDataset

# 1. 定义数据生成函数
def create_random_data(num_train=2000, num_test=500):
    """生成随机的训练和测试数据"""
    X_train = torch.randn(num_train, 3, 32, 32)
    y_train = torch.randint(0, 10, (num_train,))
    X_test = torch.randn(num_test, 3, 32, 32)
    y_test = torch.randint(0, 10, (num_test,))
    train_data = TensorDataset(X_train, y_train)
    test_data = TensorDataset(X_test, y_test)
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=128, shuffle=False)
    return train_loader, test_loader

# 2. 定义模型
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 10, 3)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(10 * 15 * 15, 10)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        return self.fc(x.view(x.size(0), -1))

# 3. 定义训练函数（必须在调用前定义！）
def train_model(model, train_loader, test_loader, epochs=8, lr=0.01, name='Model'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    train_accs, test_accs = [], []
    train_losses, test_losses = [], []
    start_time = time.time()

    print(f'\n训练 {name}...')
    for epoch in range(epochs):
        model.train()
        correct = total = 0
        epoch_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            epoch_loss += loss.item()
        train_acc = correct / total
        train_accs.append(train_acc)
        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        correct = total = 0
        test_loss = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                _, preds = outputs.max(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)
                test_loss += loss.item()
        test_acc = correct / total
        test_accs.append(test_acc)
        test_losses.append(test_loss / len(test_loader))

        scheduler.step()
        print(f'Epoch {epoch+1:2d}/{epochs} | '
              f'Train: {train_acc:.3f} | Test: {test_acc:.3f}')

    elapsed = time.time() - start_time
    print(f'{name} 完成! 用时: {elapsed:.1f}s | 最终测试准确率: {test_accs[-1]:.4f}')
    return {'train_acc': train_accs, 'test_acc': test_accs,
            'train_loss': train_losses}

---
## 6. 训练: PlainNet vs ResNet

In [19]:
print('=' * 50)
print('对比实验: PlainNet vs ResNet')
print('=' * 50)

results = {}
results['PlainNet20'] = train_model(PlainNet20(), train_loader, test_loader,
                                     epochs=5, name='PlainNet-20')
results['PlainNet32'] = train_model(PlainNet32(), train_loader, test_loader,
                                     epochs=5, name='PlainNet-32')

对比实验: PlainNet vs ResNet

训练 PlainNet-20...
Epoch  1/5 | Train: 0.099 | Test: 0.090
Epoch  2/5 | Train: 0.329 | Test: 0.102
Epoch  3/5 | Train: 0.413 | Test: 0.146
Epoch  4/5 | Train: 0.917 | Test: 0.134
Epoch  5/5 | Train: 1.000 | Test: 0.122
PlainNet-20 完成! 用时: 72.3s | 最终测试准确率: 0.1220

训练 PlainNet-32...
Epoch  1/5 | Train: 0.103 | Test: 0.090
Epoch  2/5 | Train: 0.189 | Test: 0.112
Epoch  3/5 | Train: 0.503 | Test: 0.112
Epoch  4/5 | Train: 0.748 | Test: 0.088
Epoch  5/5 | Train: 0.991 | Test: 0.100
PlainNet-32 完成! 用时: 134.1s | 最终测试准确率: 0.1000


In [20]:
results['ResNet18'] = train_model(ResNet18(), train_loader, test_loader,
                                   epochs=5, name='ResNet-18')
results['ResNet34'] = train_model(ResNet34(), train_loader, test_loader,
                                   epochs=5, name='ResNet-34')


训练 ResNet-18...
Epoch  1/5 | Train: 0.096 | Test: 0.096
Epoch  2/5 | Train: 0.300 | Test: 0.080
Epoch  3/5 | Train: 0.407 | Test: 0.116
Epoch  4/5 | Train: 0.932 | Test: 0.070
Epoch  5/5 | Train: 1.000 | Test: 0.080
ResNet-18 完成! 用时: 73.3s | 最终测试准确率: 0.0800

训练 ResNet-34...
Epoch  1/5 | Train: 0.092 | Test: 0.090
Epoch  2/5 | Train: 0.188 | Test: 0.090
Epoch  3/5 | Train: 0.436 | Test: 0.116
Epoch  4/5 | Train: 0.770 | Test: 0.124
Epoch  5/5 | Train: 0.994 | Test: 0.108
ResNet-34 完成! 用时: 134.9s | 最终测试准确率: 0.1080


---
## 7. 可视化对比

In [28]:
import matplotlib.pyplot as plt

# 定义颜色
colors = {
    'ResNet18': '#6c5ce7', 'ResNet34': '#a29bfe',
    'PlainNet20': '#e17055', 'PlainNet32': '#d63031'
}

# 【修改点】改为 1行3列，因为少了一张图
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. 训练 Loss (对应 axes[0])
ax = axes[0]
for name, res in results.items():
    ax.plot(range(1, len(res['train_loss'])+1), res['train_loss'],
            'o-', color=colors.get(name, '#333'), label=name)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('训练 Loss'); ax.legend(); ax.grid(True, alpha=0.3)

# 2. 训练准确率 (对应 axes[1])
ax = axes[1]
for name, res in results.items():
    ax.plot(range(1, len(res['train_acc'])+1), res['train_acc'],
            'o-', color=colors.get(name, '#333'), label=name)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('训练集准确率'); ax.legend(); ax.grid(True, alpha=0.3)

# 3. 测试准确率 (对应 axes[2])
ax = axes[2]
for name, res in results.items():
    ax.plot(range(1, len(res['test_acc'])+1), res['test_acc'],
            'o-', color=colors.get(name, '#333'), label=name)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('测试集准确率'); ax.legend(); ax.grid(True, alpha=0.3)

# 添加总标题并显示
plt.suptitle('PlainNet vs ResNet 对比 (无测试Loss数据)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 打印最终结果表格
print(f'{"="*40}')
print(f'{"模型":15s} {"测试准确率":>12s}')
print(f'{"="*40}')
for name in ['PlainNet20', 'PlainNet32', 'ResNet18', 'ResNet34']:
    if name in results:
        acc = results[name]['test_acc'][-1]
        print(f'{name:15s} {acc:>12.4f}')
print(f'{"="*40}')

D:\hyy\Temp\ipykernel_7816\1602365390.py:38: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
D:\hyy\Temp\ipykernel_7816\1602365390.py:38: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
D:\hyy\Temp\ipykernel_7816\1602365390.py:38: UserWarning: Glyph 38598 (\N{CJK UNIFIED IDEOGRAPH-96C6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
D:\hyy\Temp\ipykernel_7816\1602365390.py:38: UserWarning: Glyph 20934 (\N{CJK UNIFIED IDEOGRAPH-51C6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
D:\hyy\Temp\ipykernel_7816\1602365390.py:38: UserWarning: Glyph 30830 (\N{CJK UNIFIED IDEOGRAPH-786E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
D:\hyy\Temp\ipykernel_7816\1602365390.py:38: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
D:\hyy\Temp\ipykernel_7816\1602365390.py:38: UserWar

模型                     测试准确率
PlainNet20            0.1220
PlainNet32            0.1000
ResNet18              0.0800
ResNet34              0.1080


---
## 8. 为什么残差连接能解决退化?

### 梯度视角

反向传播时:

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \left(1 + \frac{\partial F}{\partial x}\right)
$$

```
传统网络:  梯度 = ∂L/∂y * ∂F/∂x        <- 如果 ∂F/∂x≈0, 梯度消失!
残差网络:  梯度 = ∂L/∂y * (1 + ∂F/∂x)  <- 即使 ∂F/∂x=0, 梯度=∂L/∂y, 永不消失!
```

### 前向视角

$$
x_{L} = x_{l} + \sum_{i=l}^{L-1} F(x_i, W_i)
$$

信息可以直接从第 $l$ 层传到第 $L$ 层, 不需要经过中间的权重矩阵。

### 一句话总结

> **残差连接 = 梯度的高速公路 + 信息的直通车**

---
## 9. 进阶: Pre-Activation ResNet

把 BN+ReLU 移到卷积前, 让恒等路径完全畅通:

```
原始:     x -> Conv -> BN -> ReLU -> Conv -> BN -> + -> ReLU
Pre-Act:  x -> BN -> ReLU -> Conv -> BN -> ReLU -> Conv -> +
```

为什么? 恒等路径上的 ReLU 会阻塞负值信号, Pre-Act 移除了这个阻塞。

In [29]:
class PreActBlock(nn.Module):
    """Pre-Activation 残差块: BN+ReLU 放卷积前"""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
            )

    def forward(self, x):
        out = F.relu(self.bn1(x))  # 先 BN+ReLU, 再卷积!
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out += self.shortcut(x)
        return out  # 最后没有 ReLU! 恒等路径完全畅通

print('Pre-Activation BasicBlock 创建成功')
print('恒等路径完全不受 ReLU 阻塞, 训练 1000 层都没问题!')

Pre-Activation BasicBlock 创建成功
恒等路径完全不受 ReLU 阻塞, 训练 1000 层都没问题!


---

# Day 13 完成!

## 今天学到了什么

| 概念 | 一句话 |
|------|--------|
| **退化问题** | 深度增加 -> 优化困难 -> 56层不如20层 |
| **残差连接** | $F(x) + x$, 学残差比学恒等容易得多 |
| **梯度高速路** | 梯度 = $\partial L/\partial y \cdot (1 + \partial F/\partial x)$, 永不消失 |
| **BasicBlock** | 两个 3x3, 用于 ResNet18/34 |
| **Bottleneck** | 1x1->3x3->1x1, 压缩-扩展, 用于 ResNet50+ |
| **GAP** | 全局平均池化代替全连接, 参数暴减90% |

## CNN 进化路线 (完整)

```
LeNet    (1998)   7层   60K     卷积+池化+全连接    开山之作
AlexNet  (2012)   8层   60M     ReLU+Dropout+GPU   深度学习革命
VGG      (2014)  16层  138M    全部3x3             深而简单
ResNet   (2015) 152层   25M    残差连接+GAP        解决退化 <- 今天
```

## 作业

1. 对比 ResNet18 vs ResNet34 vs ResNet50 的参数量和训练曲线
2. 观察 PlainNet32 是否比 PlainNet20 差 (退化现象)
3. **进阶**: 实现 PreActResNet, 对比训练曲线是否更稳定

> 预习 Day 14: **迁移学习** - 用预训练 ResNet 微调自己的数据集